# 04 · Feature engineering

Goal: give the model what a sharp handicapper looks at. How good is this offense **after
adjusting for who it played**, how good is the opponent's defense (adjusted the same way),
and what's the context?

**Golden rule (as-of):** a game's features only use games from earlier slates (a slate is
season + week). `tests/test_features.py` enforces it: hiding every future result must not
change a single feature value.

All logic lives in `src/canes_cfb/features.py`. This notebook builds the table, then
measures **which feature families actually improve the model**, and whether there are
interactions and nonlinearities for the model to capture.

Research behind the choices: `docs/research/feature_engineering.md`.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from canes_cfb.features import (
    FEATURE_SETS,
    SLATE,
    build_features,
    cumulative_sets,
    ridge_ratings,
    team_games,
)
from canes_cfb.paths import PROCESSED, RAW
from canes_cfb.validation import walk_forward

games = pd.read_parquet(RAW / "games.parquet")

## 1. Tune the opponent-adjusted ratings

`points = intercept + off[team] − def[opp] + home_edge × hfa`, fit with ridge regression
before every slate on the prior ~600 days, recency-weighted (half-life in days). Last
season acts as the prior in September, and current form takes over as games accumulate.

Two knobs: `alpha` (shrinkage toward average) and `half_life_days`. Scored on 2017–2023,
inside the train window.

In [ ]:
tg = team_games(games)
evaluate = tg[tg.completed & tg.fbs_game & tg.season.between(2017, 2023)]


def opp_def(rr):
    return rr[[*SLATE, "team_id", "def"]].rename(columns={"team_id": "opp_id", "def": "opp_def"})


grid = []
for alpha in (0.3, 1, 3, 8):
    for half_life in (60, 120, 240):
        rr = ridge_ratings(tg, alpha=alpha, half_life_days=half_life)
        m = evaluate.merge(rr, on=[*SLATE, "team_id"]).merge(opp_def(rr), on=[*SLATE, "opp_id"])
        pred = m.intercept + m.off - m.opp_def + m.hfa_pts * m.hfa
        grid.append({"alpha": alpha, "half_life": half_life, "MAE": (m.points - pred).abs().mean()})
pd.DataFrame(grid).sort_values("MAE").head()

Best: `alpha = 1`, `half_life = 120` days (MAE ≈ 9.55), now the defaults. Too much
shrinkage (alpha 8) squeezes every team toward average. Too short a memory (60 days)
forgets last season too fast for early weeks.

## 2. Build the feature table

In [ ]:
features = build_features(games)
features.to_parquet(PROCESSED / "team_games.parquet", index=False)
print(features.shape)
{family: len(cols) for family, cols in FEATURE_SETS.items()}

| Family | What | Why |
|---|---|---|
| `raw_form` | Points for/against, season-to-date and last 3, relative to league average | The naive version. Era-normalized because scoring fell ~5 points since 2015. |
| `ratings` | Ridge offense/defense for team and opponent + `exp_points` | Opponent adjustment: 40 against a bad defense ≠ 40 against a good one |
| `elo` | 538-style Elo, log margin-of-victory, offseason regression | A second rating that discounts blowouts nonlinearly |
| `matchup` | `off × opp_def`, expected margin, |margin|, margin², share of total, residual form (last 3 vs expectation), volatility | Explicit interactions and nonlinear terms, for linear models |
| `context` | Neutral, conference game, rest days and rest difference, bowl, early season, indoor, week | Situational effects |

## 3. Which feature families help?

Walk-forward CV: train on every season before, validate on 2019, 2020, 2021, 2022 and 2023.
Families are added one at a time. Ridge (linear) and LightGBM (trees, not tuned yet).

In [ ]:
data = features[features.completed & features.season.between(2016, 2023) & ~features.shortened]
data = data.reset_index(drop=True)
y = data.points.to_numpy()


def ridge():
    return make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), Ridge(alpha=1.0)
    )


def gbm(**kw):
    params = dict(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=100,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        verbose=-1,
    )
    return lgb.LGBMRegressor(**{**params, **kw})


def cv_mae(make_model, cols, target=None, offset=None):
    errors = []
    for _, train, valid in walk_forward(data):
        t = y if target is None else target
        model = make_model().fit(data.loc[train, cols], t[train])
        pred = model.predict(data.loc[valid, cols]) + (0 if offset is None else offset[valid])
        errors.append(np.abs(y[valid] - pred).mean())
    return np.mean(errors)


rows = [
    ("baseline: league average", "-", np.abs(y - data.league_avg)[data.season >= 2019].mean()),
    ("ratings formula only", "-", np.abs(y - data.exp_points)[data.season >= 2019].mean()),
]
for name, cols in cumulative_sets().items():
    rows.append((name, "ridge", cv_mae(ridge, cols)))
    rows.append((name, "lightgbm", cv_mae(gbm, cols)))
ablation = pd.DataFrame(rows, columns=["features", "model", "CV MAE"])
ablation.round(3)

**Reading it:**
- Opponent-adjusted ratings are the big win: **11.1 → ~9.45 MAE** (−15%). Raw form alone
  only gets to ~10.0, so the adjustment is worth ~0.55 points of error by itself.
- Elo, matchup and context add almost nothing on top (< 0.01).
- Untuned LightGBM is *worse* than Ridge. Nothing nonlinear is left for it to find, so
  its extra flexibility turns into noise.

## 4. Are there interactions to capture?

A direct test: LightGBM restricted to single-split trees can't model interactions (it's an
additive model). If allowing deeper trees doesn't help, the features don't contain
interactions a model can exploit.

In [ ]:
cols = cumulative_sets()["+context"]
additive = lambda: gbm(n_estimators=1500, num_leaves=2)  # noqa: E731
depth_2 = lambda: gbm(n_estimators=800, num_leaves=4, max_depth=2)  # noqa: E731
pd.Series(
    {
        "additive (stumps, no interactions)": cv_mae(additive, cols),
        "2-way interactions (depth 2)": cv_mae(depth_2, cols),
        "deep interactions (15 leaves)": cv_mae(gbm, cols),
        "boosting on residual of ratings": cv_mae(
            lambda: gbm(n_estimators=300, learning_rate=0.02, num_leaves=7, min_child_samples=200),
            cols,
            target=y - data.exp_points.to_numpy(),
            offset=data.exp_points.to_numpy(),
        ),
    }
).round(3)

**No exploitable interactions in score-only features:** additive ≈ 9.46, and deeper
trees get worse. Boosting on the ratings' residual is marginally best (~9.43). That's the
pattern to keep: the ratings carry the signal, and the model corrects what they miss.

## 5. Where is the ratings formula wrong? (nonlinearity)

In [ ]:
check = data[data.season >= 2019].assign(resid=lambda d: d.points - d.exp_points)
by_expected = check.groupby(
    pd.cut(check.exp_points, [0, 17, 21, 25, 29, 33, 37, 60]), observed=True
).resid.agg(["mean", "count"])
by_week = check.groupby(pd.cut(check.week, [0, 2, 4, 8, 20]), observed=True).resid.agg(
    lambda s: s.abs().mean()
)
display(by_expected.round(2).T, by_week.round(2).rename("MAE by week"))

- **Regression to the mean:** when the ratings expect ≤17 points, teams score ~1.5
  *more*; when they expect ≥37, ~1.2 *less*. Extreme ratings are too extreme. A linear
  model with `exp_points` as a feature fixes this with its slope, which is part of why
  Ridge ≈ formula.
- **Early season is the weak spot:** MAE ~10.0 in weeks 1–2 vs ~9.2 in weeks 5–8. Better
  priors (talent, returning production) would help most here.

## 6. Conclusions and what's next

1. Opponent adjustment is the core feature, as expected. It's already built and tuned.
2. With **only final scores**, there are no interactions left to learn. Interactions in
   football come from **how** teams score, and that needs play-level data from CFBD:
   - **Pace × efficiency**: points = plays × points per play (multiplicative). Two fast
     teams inflate totals beyond the sum of their parts.
   - **Pass/rush matchups**: a strong passing offense against a weak pass defense. This
     is the real "offense vs. defense" interaction.
   - **Weather thresholds**: wind above ~15 mph cuts scoring nonlinearly.
   - **Talent and returning production**: priors for the weak early-season weeks.
3. So the bottleneck is data, not the model: **the CFBD key (#1) is now the top priority.**

Log: `docs/experiments/2026-09-23_team_points_full_game_feature_ablation.md`.